# Prepare csv file to upload on Rubin/LSST portal

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **creation date**: 2026-06-24
- **last update date**: 2026-06-24 : Choose the good HIPS maps
- input : `data_SIMBAD_02/summary_visit_counts_per_star.parquet`
  (or `data_SIMBAD_01/master_stable_stars_V17-20_r1.5deg.csv`)

https://dp1.lsst.io/tutorials/portal/101/portal-101-2.html

## 1. Imports & configuration

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, PowerNorm

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
import astroquery

pd.set_option("display.max_rows", 50)

warnings.filterwarnings("ignore")

print(f"numpy   version : {np.__version__}")
print(f"pandas  version : {pd.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# ── Input catalogues ─────────────────────────────────────────────────────────
# Priority: summary table from NB02 (has visit counts); fallback to NB01 master
PATH_SUMMARY = "data_SIMBAD_02/summary_visit_counts_per_star.parquet"
PATH_MASTER_CSV = "data_SIMBAD_01/master_stable_stars_V17-20_r1.5deg.csv"

# ── Output directories ─────────────────────────────────────────────────────────
NB_TAG = "SIMBAD_04"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"

os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)

print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": False,
        "axes.spines.top": True,
        "axes.spines.right": True,
        "font.size": 8,
    }
)


def savefig(name: str) -> None:
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

# ── Selection: how many stars to show per DDF ─────────────────────────────────
N_STARS_PER_DDF = 30  # [adjustable] top-N by visit count per DDF
MIN_VISITS = 1  # [adjustable] skip stars with fewer total visit

## 3. Helper functions for HiPS2FITS retrieval

## 4. Load the stable star catalogue

We load the summary table from notebook `02` (which contains per-star visit
counts and coordinates).  If it is unavailable, we fall back to the Simbad
master catalogue from notebook `01`.


In [ ]:
if os.path.exists(PATH_SUMMARY):
    df_stars = pd.read_parquet(PATH_SUMMARY)
    print(f"Loaded summary table from NB02 : {df_stars.shape}")
    # Ensure ra/dec column names match NB01 convention
    if "ra_deg" not in df_stars.columns and "ra" in df_stars.columns:
        df_stars.rename(columns={"ra": "ra_deg", "dec": "dec_deg"}, inplace=True)
elif os.path.exists(PATH_MASTER_CSV):
    df_stars = pd.read_csv(PATH_MASTER_CSV)
    print(f"Loaded master catalogue from NB01 : {df_stars.shape}")
    # Add a dummy n_visits_total if absent
    if "n_visits_total" not in df_stars.columns:
        df_stars["n_visits_total"] = 0
else:
    raise FileNotFoundError(
        f"Neither {PATH_SUMMARY} nor {PATH_MASTER_CSV} found.\n" "Run notebooks 01 and 02 first."
    )

print(f"Columns : {list(df_stars.columns)}")
df_stars.head(5)

In [ ]:
# ── Select stars with known spectral type and minimum visit count ──────────────
def has_known_sptype(sp) -> bool:
    if pd.isna(sp):
        return False
    return str(sp).strip() not in ("", "?", "unknown", "--", "nan", "None")


mask = df_stars["spectral_type"].apply(has_known_sptype) & (df_stars["n_visits_total"] >= MIN_VISITS)
df_sel = df_stars[mask].copy().reset_index(drop=True)
print(f"Stars with known SpT and >= {MIN_VISITS} visits : {len(df_sel)} / {len(df_stars)}")

# Per-DDF: keep top-N by visit count
if "field" in df_sel.columns:
    df_targets = (
        df_sel.sort_values("n_visits_total", ascending=False)
        .groupby("field")
        .head(N_STARS_PER_DDF)
        .reset_index(drop=True)
    )
else:
    df_targets = df_sel.head(N_STARS_PER_DDF * 4).reset_index(drop=True)

print(f"Target stars for cutouts : {len(df_targets)}")
cols_show = ["simbad_id", "spectral_type", "V_mag", "ra_deg", "dec_deg", "field", "n_visits_total"]
cols_show = [c for c in cols_show if c in df_targets.columns]
df_targets[cols_show]

## Save files

In [ ]:
list_of_fields = df_targets["field"].unique()
print(list_of_fields)

In [ ]:
for field in list_of_fields:
    selecting_cut = df_targets["field"] == field
    df = df_targets[["simbad_id", "ra_deg", "dec_deg"]][selecting_cut]
    df.rename(columns={"ra_deg": "ra", "dec_deg": "dec"}, inplace=True)
    csv_out = os.path.join(DIR_DATA, f"stable_stars_DDF{field}.csv")
    df.to_csv(csv_out, index=False)

In [ ]:
df